# Dev / Test / Prod Environments (Serverless Model) — RouteMind

**Scenario:** RouteMind is about to move from "a notebook on someone's laptop" to a real deployment behind FastAPI, and that means Portside Logistics needs separate dev, test, and production configurations before it ships. This notebook introduces environment management and testing patterns for that transition.

Topics:
- Environment-specific `.env` files
- Development, Test, and Production configurations
- In-memory vs file-backed SQLite
- Pytest fixtures
- Test data seeding
- Test teardown
- Local evaluation workflows

The focus is on lightweight local/serverless development before introducing cloud deployments.

## Learning Goals

By the end of this notebook you should be able to:

1. Manage environment-specific configuration.
2. Separate dev, test, and production settings.
3. Use SQLite differently across environments.
4. Seed databases for testing.
5. Clean up test artifacts automatically.
6. Build repeatable local evaluation workflows.

In [ ]:
%pip install -q python-dotenv pytest sqlmodel

## 1) Environment Strategy

A common layout for RouteMind:

```text
routemind/
│
├── .env.dev
├── .env.test
├── .env.prod
│
├── data/
│   ├── routemind_dev.db
│   ├── routemind_test.db
│   └── routemind_prod.db
│
├── tests/
└── src/
```

Each environment has its own settings, and each Portside depot ultimately points at the same `prod` config, just a different row of data.

In [ ]:
from dotenv import load_dotenv
import os

APP_ENV = os.getenv("APP_ENV", "dev")

env_map = {
    "dev": ".env.dev",
    "test": ".env.test",
    "prod": ".env.prod"
}

load_dotenv(env_map.get(APP_ENV, ".env.dev"))

print("Environment:", APP_ENV)

## 2) Example Environment Files

### .env.dev

```env
APP_ENV=dev
DB_PATH=data/routemind_dev.db
LOG_LEVEL=DEBUG
```

### .env.test

```env
APP_ENV=test
DB_PATH=:memory:
LOG_LEVEL=INFO
```

### .env.prod

```env
APP_ENV=prod
DB_PATH=data/routemind_prod.db
LOG_LEVEL=WARNING
```

## 3) SQLite Lifecycle

Development:
- Persistent database file, one dispatcher testing changes locally

Testing:
- In-memory database, wiped clean on every CI run

Production:
- Persistent production database, one per depot deployment

This keeps tests isolated and fast, so a bug in a test run can never touch real shipment data.

In [ ]:
import sqlite3

# Test database
test_conn = sqlite3.connect(":memory:")

test_conn.execute(
    '''
    CREATE TABLE routemind_prompt_log(
        id INTEGER PRIMARY KEY,
        prompt TEXT
    )
    '''
)

print("In-memory database ready")

In [ ]:
# Production-style database

prod_conn = sqlite3.connect("routemind_prod.db")

prod_conn.execute(
    '''
    CREATE TABLE IF NOT EXISTS routemind_prompt_log(
        id INTEGER PRIMARY KEY,
        prompt TEXT
    )
    '''
)

prod_conn.commit()

print("Persistent database ready")

## 4) Environment-Aware Database Factory

Centralize connection creation so RouteMind's code never hardcodes a path -- it just asks for "the current environment's database" and gets the right one.

In [ ]:
import sqlite3
import os

def get_db_connection():

    db_path = os.getenv("DB_PATH", "routemind_dev.db")

    return sqlite3.connect(db_path)

conn = get_db_connection()

print(conn)

## 5) Why In-Memory Databases for Tests?

Benefits:

- Fast
- No cleanup required
- Isolated
- Repeatable
- No accidental production data modification -- a broken test can never touch a real depot's shipment history

## 6) Pytest Fixtures

Fixtures create reusable setup and teardown logic.

Typical flow:

Setup
    ↓
Seed Data
    ↓
Run Tests
    ↓
Cleanup

In [ ]:
# tests/conftest.py

import pytest
import sqlite3

@pytest.fixture
def db_connection():

    conn = sqlite3.connect(":memory:")

    conn.execute(
        '''
        CREATE TABLE routemind_prompt_log(
            id INTEGER PRIMARY KEY,
            prompt TEXT
        )
        '''
    )

    yield conn

    conn.close()

## 7) Seeding Test Data

Seed realistic dispatcher questions before running tests.

In [ ]:
def seed_prompts(conn):

    prompts = [
        ("What's the ETA for shipment SHIP-1042?",),
        ("How many shipments are delayed in Chennai this week?",),
        ("What does the SLA say about delay compensation?",)
    ]

    conn.executemany(
        "INSERT INTO routemind_prompt_log(prompt) VALUES (?)",
        prompts
    )

    conn.commit()

In [ ]:
# Example test

def test_prompt_count(db_connection):

    seed_prompts(db_connection)

    cursor = db_connection.execute(
        "SELECT COUNT(*) FROM routemind_prompt_log"
    )

    count = cursor.fetchone()[0]

    assert count == 3

## 8) File-Based Test Databases

Sometimes you need persistence during tests -- for example, a test that checks RouteMind's cache survives a process restart.

Example:

```text
tests/
└── routemind_test.db
```

Create it before the test and delete it afterward.

In [ ]:
from pathlib import Path

TEST_DB = Path("routemind_test.db")

if TEST_DB.exists():
    TEST_DB.unlink()

print("Clean test state")

In [ ]:
# Fixture with cleanup

import pytest
import sqlite3

@pytest.fixture
def file_db():

    conn = sqlite3.connect("routemind_test.db")

    yield conn

    conn.close()

    Path("routemind_test.db").unlink(missing_ok=True)

## 9) Mock Vector Store Seeding

Before wiring up Chroma and full RAG evaluation, you can simulate vector records for the driver handbook and SLA policy.

In [ ]:
mock_vectors = [
    {
        "id": "1",
        "text": "Drivers must take a 30 minute break after 5 hours of continuous driving."
    },
    {
        "id": "2",
        "text": "Standard service targets 3 to 5 business days between origin depot and destination city."
    }
]

print(mock_vectors)

## 10) Evaluation Workflow

Simple local workflow:

1. Seed relational data (shipments, drivers, customers)
2. Seed vector data (handbook and policy snippets)
3. Run retrieval tests
4. Verify outputs
5. Cleanup

This pattern scales nicely into full RAG evaluations later, once the driver handbook and SLA policy are actually chunked and embedded.

In [ ]:
def evaluate():

    expected = "30 minute break"

    response = "Drivers must take a 30 minute break after 5 hours of continuous driving."

    assert expected in response

    return "Evaluation Passed"

print(evaluate())

## 11) CI/CD Perspective

A typical pipeline for RouteMind:

Developer Push
        ↓
Unit Tests
        ↓
Seed Test Data
        ↓
Evaluation Tests
        ↓
Cleanup
        ↓
Deploy

LangSmith CI/CD examples use a similar philosophy of automated validation before deployment.

## 12) Recommended Local Strategy

Development:
- routemind_dev.db

Testing:
- :memory:

Production:
- routemind_prod.db (one per depot deployment)

Benefits:
- Simplicity
- Fast testing
- Safe isolation
- Easy debugging

## 13) Mini Exercise

1. Create `.env.dev`, `.env.test`, and `.env.prod` for RouteMind.
2. Build a database factory like the one in section 4.
3. Add a `routemind_token_usage` table alongside `routemind_prompt_log`.
4. Create pytest fixtures for both tables.
5. Seed sample dispatcher questions.
6. Verify row counts.
7. Clean up test artifacts.

## Key Takeaways

- Keep configuration environment-specific.
- Use python-dotenv for loading settings.
- Use `:memory:` SQLite for tests.
- Use file-backed SQLite for persistent environments.
- Use pytest fixtures for setup and teardown.
- Seed realistic test data -- real dispatcher questions, not placeholder text.
- Remove test artifacts automatically.
- Build repeatable evaluation workflows before moving into RAG and agents.

## References

- python-dotenv: https://pypi.org/project/python-dotenv/
- pytest fixtures: https://docs.pytest.org/en/stable/how-to/fixtures.html
- SQLite in-memory databases: https://www.sqlite.org/inmemorydb.html
- SQLModel: https://sqlmodel.tiangolo.com/